In [ ]:
# Use magic so package install works in notebooks
%pip install tensorflow pillow matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Chunk 1/10 - Imports + global config
import os
import json
import random
from collections import Counter

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger,
)

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
TRAIN_EPOCHS_HEAD = 12
TRAIN_EPOCHS_FINETUNE = 25
UNFREEZE_LAST_N = 40

DATASET_CANDIDATES = [r"C:\Users\Acer\OneDrive\Desktop\Artefact\dataset"]
OUTPUT_DIR = r"C:\Users\Acer\TG"

DATASET_PATH = next((p for p in DATASET_CANDIDATES if os.path.isdir(p)), None)
if DATASET_PATH is None:
    raise FileNotFoundError("Dataset folder not found in expected locations.")

os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(OUTPUT_DIR, "tomato_disease_model.keras")
CLASS_MAP_PATH = os.path.join(OUTPUT_DIR, "model_classes.json")
TRAIN_META_PATH = os.path.join(OUTPUT_DIR, "training_meta.json")
HISTORY_CSV_PATH = os.path.join(OUTPUT_DIR, "training_history.csv")

print("Using dataset:", DATASET_PATH)
print("Model output:", MODEL_PATH)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


Using dataset: C:\Users\Acer\OneDrive\Desktop\Artefact\dataset
Model output: C:\Users\Acer\TG\tomato_disease_model.keras


In [3]:
# Chunk 2/10 - Dataset audit (class counts)
class_counts = {}
for cls_name in sorted(os.listdir(DATASET_PATH)):
    cls_path = os.path.join(DATASET_PATH, cls_name)
    if not os.path.isdir(cls_path):
        continue
    count = sum(
        1
        for f in os.listdir(cls_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    )
    class_counts[cls_name] = count

if not class_counts:
    raise RuntimeError("No class folders/images found in dataset path.")

print("Class distribution:")
for name, count in class_counts.items():
    print(f"  {name}: {count}")

least = min(class_counts.values())
most = max(class_counts.values())
print(f"Imbalance ratio (max/min): {most}/{least} = {most / max(least, 1):.2f}")


Class distribution:
  Tomato_Early_blight: 1000
  Tomato_Healthy: 1591
  Tomato_Late_blight: 1909
  Tomato_Septoria_leaf_spot: 1771
Imbalance ratio (max/min): 1909/1000 = 1.91


In [4]:
# Chunk 3/10 - Data generators (production-aligned preprocessing)
# IMPORTANT: For MobileNetV2, use preprocess_input in both train and validation.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    horizontal_flip=True,
    validation_split=0.2,
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

num_classes = len(train_generator.class_indices)
print("Classes:", train_generator.class_indices)

if num_classes < 2:
    raise RuntimeError("At least 2 classes are required for training.")


Found 5018 images belonging to 4 classes.
Found 1253 images belonging to 4 classes.
Classes: {'Tomato_Early_blight': 0, 'Tomato_Healthy': 1, 'Tomato_Late_blight': 2, 'Tomato_Septoria_leaf_spot': 3}


In [7]:
# Chunk 4/10 - Persist class map + class weights
index_to_class = {str(v): k for k, v in train_generator.class_indices.items()}
with open(CLASS_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(index_to_class, f, indent=2)
print("Saved class mapping to:", CLASS_MAP_PATH)

y_train = train_generator.classes
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weight)

train_class_hist = Counter(y_train.tolist())
val_class_hist = Counter(val_generator.classes.tolist())
print("Train class index histogram:", dict(train_class_hist))
print("Val class index histogram:", dict(val_class_hist))


Saved class mapping to: C:\Users\Acer\TG\model_classes.json
Class weights: {0: 1.568125, 1: 0.9854673998428908, 2: 0.8210078534031413, 3: 0.8853211009174312}
Train class index histogram: {0: 800, 1: 1273, 2: 1528, 3: 1417}
Val class index histogram: {0: 200, 1: 318, 2: 381, 3: 354}


In [8]:
# Chunk 5/10 - Build model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)

for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(num_classes, activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,586,948 (9.87 MB)

 Trainable params: 328,964 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
# Chunk 6/10 - Callbacks
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True),
    CSVLogger(HISTORY_CSV_PATH, append=False),
]

print("Callbacks ready. Best model checkpoint:", MODEL_PATH)
print("Training log CSV:", HISTORY_CSV_PATH)


Callbacks ready. Best model checkpoint: C:\Users\Acer\TG\tomato_disease_model.keras
Training log CSV: C:\Users\Acer\TG\training_history.csv


In [10]:
# Chunk 7/10 - Stage 1 training (head only)
history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=TRAIN_EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=callbacks,
)


Epoch 1/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.8045 - loss: 0.5265 - val_accuracy: 0.9106 - val_loss: 0.2346 - learning_rate: 0.0010
Epoch 2/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 213s 1s/step - accuracy: 0.9002 - loss: 0.2882 - val_accuracy: 0.9250 - val_loss: 0.2079 - learning_rate: 0.0010
Epoch 3/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 265s 2s/step - accuracy: 0.9139 - loss: 0.2552 - val_accuracy: 0.9314 - val_loss: 0.1806 - learning_rate: 0.0010
Epoch 4/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 214s 1s/step - accuracy: 0.9189 - loss: 0.2279 - val_accuracy: 0.9338 - val_loss: 0.1796 - learning_rate: 0.0010
Epoch 5/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 214s 1s/step - accuracy: 0.9217 - loss: 0.2185 - val_accuracy: 0.9098 - val_loss: 0.2225 - learning_rate: 0.0010
Epoch 6/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 85s 542ms/step - accuracy: 0.9308 - loss: 0.1945 - val_accuracy: 0.9385 - val_loss: 0.1657 - learning_rate: 0.0010
Epoch 7/12
157/157 ━━━━━━━━━━━━━━━━━━━━ 207s 1s/step - accuracy: 0.9342 - loss: 

In [11]:
# Chunk 8/10 - Stage 2 fine-tuning
for layer in base_model.layers[-UNFREEZE_LAST_N:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=TRAIN_EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=callbacks,
)


Epoch 1/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 111s 608ms/step - accuracy: 0.9554 - loss: 0.1275 - val_accuracy: 0.9561 - val_loss: 0.1099 - learning_rate: 1.0000e-05
Epoch 2/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 93s 594ms/step - accuracy: 0.9657 - loss: 0.1075 - val_accuracy: 0.9609 - val_loss: 0.1054 - learning_rate: 1.0000e-05
Epoch 3/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 85s 538ms/step - accuracy: 0.9657 - loss: 0.0965 - val_accuracy: 0.9617 - val_loss: 0.1023 - learning_rate: 1.0000e-05
Epoch 4/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 420s 3s/step - accuracy: 0.9685 - loss: 0.0935 - val_accuracy: 0.9633 - val_loss: 0.1034 - learning_rate: 1.0000e-05
Epoch 5/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 93s 589ms/step - accuracy: 0.9695 - loss: 0.0906 - val_accuracy: 0.9689 - val_loss: 0.0939 - learning_rate: 1.0000e-05
Epoch 6/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 105s 670ms/step - accuracy: 0.9745 - loss: 0.0770 - val_accuracy: 0.9665 - val_loss: 0.1043 - learning_rate: 1.0000e-05
Epoch 7/25
157/157 ━━━━━━━━━━━━━━━━━━━━ 92s 58

In [14]:
# Chunk 9/10 - Evaluation on best checkpoint
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Best checkpoint loaded from:", MODEL_PATH)

val_generator.reset()
y_prob = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = val_generator.classes
class_names = [
    name for name, _ in sorted(val_generator.class_indices.items(), key=lambda kv: kv[1])
]

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

pred_hist = Counter(y_pred.tolist())
print("Predicted index histogram:", dict(pred_hist)) 


Best checkpoint loaded from: C:\Users\Acer\TG\tomato_disease_model.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 228ms/step
                           precision    recall  f1-score   support

      Tomato_Early_blight     0.9296    0.9250    0.9273       200
           Tomato_Healthy     0.9907    1.0000    0.9953       318
       Tomato_Late_blight     0.9840    0.9711    0.9775       381
Tomato_Septoria_leaf_spot     0.9748    0.9831    0.9789       354

                 accuracy                         0.9745      1253
                macro avg     0.9698    0.9698    0.9698      1253
             weighted avg     0.9744    0.9745    0.9744      1253

Confusion matrix:
 [[185   1   5   9]
 [  0 318   0   0]
 [ 11   0 370   0]
 [  3   2   1 348]]
Predicted index histogram: {3: 357, 0: 199, 1: 321, 2: 376}


In [15]:
# Chunk 10/10 - Deployment metadata + load verification
training_meta = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "preprocess_mode": "mobilenet_v2",
    "class_map_path": CLASS_MAP_PATH,
    "model_path": MODEL_PATH,
}

with open(TRAIN_META_PATH, "w", encoding="utf-8") as f:
    json.dump(training_meta, f, indent=2)

print("Saved deployment metadata:", TRAIN_META_PATH)
print("Saved model:", os.path.exists(MODEL_PATH))
print("Saved class map:", os.path.exists(CLASS_MAP_PATH))
print("Saved history CSV:", os.path.exists(HISTORY_CSV_PATH))

loaded = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Loaded model output shape:", loaded.output_shape)


Saved deployment metadata: C:\Users\Acer\TG\training_meta.json
Saved model: True
Saved class map: True
Saved history CSV: True
Loaded model output shape: (None, 4)
